In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from scipy.stats import wilcoxon

from project.utils.data import load_hillshade, sel_subset
from project.utils.metrics import compute_metrics
from project.utils.paths import get_path
from project.utils.plotting import ProjectCustomExtent, despine

plt.rcParams["figure.dpi"] = 150

# Switzerland LV95 with extended limits to cover 'Hydrological Switzerland'.
lv95 = ProjectCustomExtent(2056, [2474273, 2842000, 1058000, 1325000])

In [ ]:
# Load predictions.
ds = xr.open_zarr(get_path('predictions_file')).sel(
    time=slice('1993', None))

ds = ds.sel(model=[m for m in ds['model'].values if m.startswith('lumped_') | m.startswith('routed_') | m.startswith('instant_')])

# Load HRU polygons.
hrus = gpd.read_file(
    get_path('hru_file'), use_arrow=True).set_index('EZGNR')

metrics_max = ['kge', 'r']
metrics_min = ['kge_alpha', 'kge_beta', 'flv', 'fhv', 'bias', 'varerr']

# metric_nice_names = {
#     'bias2': 'Bias²',
#     'varerr': 'Variance error',
#     'phaseerr': 'Phase error',
#     'nse': 'Nash-Sutcliffe Efficiency',
#     'nnse': 'Normalized Nash-Sutcliffe Efficiency',
#     'kge': 'Kling-Gupta Efficiency',
#     'r': 'Pearson\'s r',
#     'relbias': '%Bias',
#     'flv': '%Low Flow Bias (<Q5)',
#     'fhv': '%High Flow Bias (>Q98)',
# }

# Compute metrics and add them to dataset.
met = compute_metrics(ds.Q_obs_m3s, ds.Q_pred_m3s, metrics=metrics_min + metrics_max, dim='time')
ds = xr.merge([ds, met])

for m in metrics_min + metrics_max:
    name = f'{m}_impr'

    # Create the target variable once per metric.
    # Use the metric itself as a template (and make sure it's float so NaN works).
    if name not in ds:
        ds[name] = xr.full_like(ds[m], np.nan, dtype=float).load()

    for model in ds.model.values:
        # Pair model with the corresponding other model
        model_type, split_type = model.split('_')
        other_model_type = 'routed' if model_type == 'lumped' else 'lumped'
        other_model = f'{other_model_type}_{split_type}'
        if other_model not in ds.model.values:
            continue

        value_model = ds.sel(model=model)[m]
        value_other = ds.sel(model=other_model)[m]

        if m in metrics_min:
            value_model = np.abs(value_model)
            value_other = np.abs(value_other)

        if m in metrics_min:
            # Lower is better
            pec_impr = (value_other - value_model) / (value_other + 1e-6) * 100.0
        else:
            # Higher is better
            pec_impr = (value_model - value_other) / (1 - value_other) * 100.0

        # Write only the slice for this model
        ds[name].loc[dict(model=model)] = pec_impr

# rivers = gpd.read_file(get_path('raw_data_root') / 'raw/rivernet', use_arrow=True)
rivers = gpd.read_file(get_path('raw_data_root') / 'raw/rivernet', use_arrow=True)

rivers_subset = rivers.loc[rivers.CUM_LEN > 20000]

bg = load_hillshade(
    hillshade_dir='./data',
    dem_path=get_path('raw_data_root') / 'raw/merit.zarr',
    hrus=hrus,
    ds=ds,
    use_terrain=False,
    force_recompute=False)

# Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 2), sharey=True, sharex=True, gridspec_kw={'hspace': 0.25})

ds_m = sel_subset(ds, 3)['kge']

for j, (split_type, split_name) in enumerate(zip(['random', 'hard'], ['RS', 'CS'])):
    ax = axes[j]

    lumped_met = 0.0
    perc_impr = 0.0

    for t, model_type in enumerate(['lumped', 'instant', 'routed']):

        model = f'{model_type}_{split_type}'

        ds_m_subset = ds_m.sel(model=model)
        ds_m_subset = ds_m_subset[np.isfinite(ds_m_subset)]

        colors = {
            'lumped': "#4F9CFF",
            'instant': "#FF6347",
            'routed': "#DAA507",
        }

        color = colors[model_type]

        model_type_new = {
            'lumped': 'LSTM (lumped)',
            'instant': 'DROP (instant)',
            'routed': 'DROP (routed)'
        }[model_type]

        ax.ecdf(
            ds_m_subset,
            lw=1.2,
            color=color,
            label=f'{model_type_new}'
        )

        #ax.axvline(0, color='k', ls='--', dashes=(5, 5), lw=0.4, zorder=999)

        median = ds_m_subset.median()
        if model_type == 'lumped':
            lumped_met = median.item()
            disp_txt = f'Median={lumped_met:0.2f}'
        else:
            perc_impr = (median.item() - lumped_met) / lumped_met * 100.0
            if perc_impr > 0:
                sign = '+'
            else:
                sign = '-'
            disp_txt = f'Median={median.item():0.2f} ({sign}{perc_impr:0.0f}%)'

        ax.axvline(median, ls='--', lw=1.2, color=color)
        ax.text(0.02, [0.95, 0.85, 0.75][t], disp_txt, color=color,
                va='top', ha='left', transform=ax.transAxes, fontsize=8, weight='bold')

    if j == 0:
        ax.set_ylabel('Cumulative Density', y=0.5)
    else:
        ax.set_ylabel('')

    if j == 1:
        ax.set_xlabel(ds_m.attrs['long_name'], x=-0.1)

    ax.set_xlim(-1, 1)

    ax.set_title(f'Split type: {split_name}', fontsize=10)

    despine(ax)

axes[0].legend(
    frameon=False, handlelength=1, title='Model type', loc='center left', bbox_to_anchor=(0, 0.4),
    fontsize=8, title_fontsize=9)

fig.savefig('./figures/kge.pdf', dpi=300, bbox_inches='tight')

In [ ]:
metrics = ['r', 'kge_alpha', 'kge_beta', 'flv', 'fhv']

fig, axes = plt.subplots(1, len(metrics), figsize=(10, 2),
                         sharex=True, sharey=False,
                         gridspec_kw={'wspace': 0.4})


df = sel_subset(
    ds.where(ds.has_obs.load(), drop=True), 3)[metrics].to_dataframe(
        ).reset_index()
parts = df['model'].str.rsplit('_', n=1, expand=True)
df['Model type'] = parts[0].str.capitalize()
df['Split type'] = parts[1].str.capitalize()
df['Model type'] = df['Model type'].replace('Lumped', 'LSTM (lumped)')
df['Model type'] = df['Model type'].replace('Instant', 'DROP (instant)')
df['Model type'] = df['Model type'].replace('Routed', 'DROP (routed)')

for m, metric in enumerate(metrics):

    ax = axes[m]

    colors = {
        'LSTM (lumped)': "#4F9CFF",
        'DROP (instant)': "#E8443A",
        'DROP (routed)': "#DAA507",
    }

    # common kwargs for both panels
    box_kws = dict(
        data=df, x='Split type', y=metric, hue='Model type',
        palette=colors, showfliers=False, gap=0.2
    )

    # draw the same plot twice (bottom zoom + top zoom)
    # legend only on the top row, last metric (bottom axis) to place later
    want_legend = m == 0
    sns.boxplot(ax=ax, order=['Random', 'Hard'], hue_order=['LSTM (lumped)', 'DROP (instant)', 'DROP (routed)'], legend=want_legend, **box_kws)

    ax.set_xticklabels(['RS', 'CS'])

    metric_nice = ds[metric].attrs['long_name']

    # titles / labels
    ax.set_title(metric_nice, fontsize=8)

    ax.set_ylabel('')

    # reference line (draw on both; it will only show if within the panel range)
    # for a in (ax_bot, ax_top):
    #     a.axhline(1, color='k', ls='--', lw=0.8, zorder=0.001)

    # optional: your despine helper on both axes
    despine(ax)

    ax.axhline(1, color='k', ls='--', lw=0.8, zorder=999)

    if want_legend:
        legend_ax = ax

    if m == 2:
        ax.set_xlabel('Split type')
    else:
        ax.set_xlabel('')

# # place the legend outside using the saved axis
if legend_ax is not None:
    sns.move_legend(legend_ax, loc='upper left', ncol=3, title='', columnspacing=0.8, handletextpad=0.3,
                    bbox_to_anchor=(-0.18, -0.13), frameon=False, fontsize=8, title_fontsize=9)

fig.savefig('./figures/biases.pdf', dpi=300, bbox_inches='tight')

# Model comparison wrt catchment area

In [ ]:
# --- config: add/remove metrics freely ---
metrics = ['kge', 'r', 'kge_alpha', 'kge_beta']
setup = 'random'

# --- data ---
dss = sel_subset(ds.where(ds.has_obs.load(), drop=True), 3)
ds_lumped  = dss.sel(model=f'lumped_{setup}')
 
df_list = []
for met in metrics:
    for model_key, label in [(f'instant_{setup}',  'DROP (instant)'),
                               (f'routed_{setup}', 'DROP (routed)')]:
        ds_model = dss.sel(model=model_key)
        diff = (ds_model[met] - ds_lumped[met]) / np.abs(ds_lumped[met]) * 100
 
        area   = ds_model.upstream_area.values.astype(float)
        vals   = diff.values.astype(float)
        valid  = np.isfinite(area) & np.isfinite(vals)
 
        area_v = area[valid]
        vals_v = vals[valid]
 
        # equal-sample bins
        n_bins      = 4
        sorted_idx  = np.argsort(area_v)
        bin_assign  = np.empty(len(area_v), dtype=int)
        for i, chunk in enumerate(np.array_split(sorted_idx, n_bins)):
            bin_assign[chunk] = i
 
        def fmt_area(v):
            if v >= 1e6: return f"{v/1e6:.1f}M"
            if v >= 1e3: return f"{v/1e3:.1f}k"
            return f"{v:.0f}"
 
        for i in range(n_bins):
            mask = bin_assign == i
            lo   = area_v[mask].min()
            hi   = area_v[mask].max()
            # bin_label = f"{fmt_area(lo)}–{fmt_area(hi)}"
            bin_label = f">{fmt_area(lo)}"
            for v in vals_v[mask]:
                df_list.append({'metric': met, 'Model': label,
                                'bin': i, 'bin_label': bin_label, 'value': v})
 
import pandas as pd
df = pd.DataFrame(df_list)
 
# ordered bin labels per metric (consistent x-axis)
bin_order_map = {}
for met in metrics:
    sub = df[df['metric'] == met].drop_duplicates('bin').sort_values('bin')
    bin_order_map[met] = sub['bin_label'].tolist()
 
# --- plot ---
colors = {
    'DROP (instant)': "#E8443A",
    'DROP (routed)':  "#DAA507",
}
hue_order = ['DROP (instant)', 'DROP (routed)']

fig, axes = plt.subplots(1, len(metrics), figsize=(len(metrics) * 2, 2),
                         sharey=False,
                         gridspec_kw={'wspace': 0.25})
 
for m, met in enumerate(metrics):
    ax      = axes[m]
    sub     = df[df['metric'] == met]
    b_order = bin_order_map[met]
 
    sns.boxplot(
        data=sub, x='bin_label', y='value', hue='Model',
        order=b_order, hue_order=hue_order,
        palette=colors, showfliers=False, gap=0.2,
        legend=(m == 0),
        ax=ax,
    )

    for i, bl in enumerate(b_order):
        n = sub[sub['bin_label'] == bl]['value'].count() // 2
        ax.text((i + 0.5) / len(b_order), 1.04, f"n={n}",
                ha='center', va='top', fontsize=6, color='#666',
                transform=ax.transAxes)

    ax.axhline(0, color='k', ls='--', lw=0.8, zorder=999)
 
    metric_nice = ds[met].attrs.get('long_name', met)
    ax.set_title(metric_nice, fontsize=8)
    ax.set_ylabel('% improvement vs.\nLSTM (lumped)' if m == 0 else '', fontsize=7, labelpad=0)
    ax.set_xlabel('Upstream area (km²)' if m == len(metrics) // 2 else '', fontsize=7, x=-0.25)
    ax.tick_params(axis='x', labelsize=6.5)
    ax.tick_params(axis='y', labelsize=7)
    despine(ax)
 
    if m == 0:
        legend_ax = ax

for ax in axes:
    ylo, yhi = ax.get_ylim()
    abs_max = max(abs(ylo), abs(yhi))
    ax.set_ylim(-abs_max, abs_max)

# legend below figure
if legend_ax is not None:
    sns.move_legend(
        legend_ax,
        loc='upper left',
        ncol=2,
        title='',
        columnspacing=0.8,
        handletextpad=0.3,
        bbox_to_anchor=(0.0, -0.1),
        frameon=False,
        fontsize=8,
    )


fig.savefig('./figures/improvement_by_area.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- config ---
metrics = ['kge', 'r', 'kge_alpha', 'kge_beta']
setup = 'random'

# --- data ---
dss = sel_subset(ds.where(ds.has_obs.load(), drop=True), 3)

df_list = []
for met in metrics:
    ds_instant = dss.sel(model=f'instant_{setup}')
    ds_routed  = dss.sel(model=f'routed_{setup}')
    
    # % improvement: (routed - instant) / |instant| * 100
    diff = (ds_routed[met] - ds_instant[met]) / np.abs(ds_instant[met]) * 100
    
    area   = ds_routed.upstream_area.values.astype(float)
    vals   = diff.values.astype(float)
    valid  = np.isfinite(area) & np.isfinite(vals)
    
    area_v = area[valid]
    vals_v = vals[valid]
    
    # equal-sample bins
    n_bins      = 4
    sorted_idx  = np.argsort(area_v)
    bin_assign  = np.empty(len(area_v), dtype=int)
    for i, chunk in enumerate(np.array_split(sorted_idx, n_bins)):
        bin_assign[chunk] = i
    
    def fmt_area(v):
        if v >= 1e6: return f"{v/1e6:.1f}M"
        if v >= 1e3: return f"{v/1e3:.1f}k"
        return f"{v:.0f}"
    
    for i in range(n_bins):
        mask = bin_assign == i
        lo   = area_v[mask].min()
        hi   = area_v[mask].max()
        bin_label = f">{fmt_area(lo)}"
        for v in vals_v[mask]:
            df_list.append({'metric': met, 'bin': i, 'bin_label': bin_label, 'value': v})

df = pd.DataFrame(df_list)

# bin order per metric
bin_order_map = {}
for met in metrics:
    sub = df[df['metric'] == met].drop_duplicates('bin').sort_values('bin')
    bin_order_map[met] = sub['bin_label'].tolist()

# --- plot ---
fig, axes = plt.subplots(1, len(metrics), figsize=(len(metrics) * 2, 2),
                         sharey=False,
                         gridspec_kw={'wspace': 0.3})

for m, met in enumerate(metrics):
    ax      = axes[m]
    sub     = df[df['metric'] == met]
    b_order = bin_order_map[met]
    
    sns.boxplot(
        data=sub, x='bin_label', y='value',
        order=b_order,
        color="0.2",
        linewidth=1,
        gap=0.2,
        showfliers=False,
        fill=False,
        ax=ax,
    )

    for i, bl in enumerate(b_order):
        n = sub[sub['bin_label'] == bl]['value'].count()
        ax.text((i + 0.5) / len(b_order), 1.0, f"n={n}",
                ha='center', va='top', fontsize=6, color='#666',
                transform=ax.transAxes)
    
    ax.axhline(0, color='k', ls='--', lw=0.8, zorder=999)
    
    metric_nice = ds[met].attrs.get('long_name', met)
    ax.set_title(metric_nice, fontsize=8, pad=5)
    ax.set_ylabel('% improvement\nDROP (routed) vs. DROP (instant)' if m == 0 else '', 
                  fontsize=7, labelpad=2)
    ax.set_xlabel('Upstream area (km²)' if m == len(metrics) // 2 else '', fontsize=7, x=-0.14)

    ax.tick_params(axis='both', labelsize=7)
    despine(ax)

# symmetric y-limits (set BEFORE plotting labels)
for ax in axes:
    ylo, yhi = ax.get_ylim()
    abs_max = max(abs(ylo), abs(yhi))
    ax.set_ylim(-abs_max * 1.15, abs_max * 1.15)  # 15% padding for labels

# re-plot labels after axis limits are finalized
for m, met in enumerate(metrics):
    ax = axes[m]
    sub = df[df['metric'] == met]
    b_order = bin_order_map[met]

fig.savefig('./figures/improvement_by_area_drop.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Overfitting analysis

To assess whether the 'random' setup overfits, we compare it with the 'hard' (='other') setup on the catchments that
are in the evaluation set (validation or test) of both setups, and compute the KGE difference 'random' minus 'other'
per catchment.

We test the differences with a Wilcoxon signed-rank test. A significant difference would indicate that the 'random'
setup is overfitting to its training set. Further, we report the median difference and the effect size.

In [ ]:
ds_r = ds.sel(model='routed_random')
ds_h = ds.sel(model='routed_hard')

In [ ]:
def overfit_test(a: xr.Dataset, b: xr.Dataset) -> None:
    """
    Test whether model 'a' overfits, using a Wilcoxon signed-rank test of its KGE difference to model 'b' on the
    catchments in the evaluation set of both.

    Args:
        a (xr.Dataset): Dataset of model 'a' (random-split trained model).
        b (xr.Dataset): Dataset of model 'b' (spatial-split 'hard' trained model).

    """

    fig, ax = plt.subplots(figsize=(3, 1.5), sharey=True, gridspec_kw={'wspace': 0.05})

    for i, (o, name) in enumerate(zip([b], ['CS'])):

        # Find HRUs which are in the training set of both models, and those which are in the evaluation set of both models.
        both_eval = ((a.subset.isin([2, 3])) & (o.subset.isin([2, 3]))).compute()

        # Relative gain model a to b for the training and evaluation sets.
        a_kge = a.kge.where(both_eval, drop=True)
        o_kge = o.kge.where(both_eval, drop=True)
        diff = a_kge - o_kge

        wc = wilcoxon(diff, method='asymptotic')

        v = wc.statistic
        p = wc.pvalue
        z = wc.zstatistic
        n = len(diff)
        r = z / np.sqrt(n)

        bins = np.linspace(-.6, .6, 12)
        ax.hist(diff, bins=bins, alpha=0.6, density=False, color='0.5')
        ax.axvline(diff.median().item(), color='tab:red', ls='--', lw=1.4, label='Median', zorder=10)
        ax.axvline(0, color='k', ls='-', lw=1.4, zorder=9)
        ax.set_xlabel('KGE difference', fontsize=8)
        if i == 0:
            ax.set_ylabel('Density', size=8)
        else:
            ax.legend(frameon=False, loc='upper right', fontsize=8)
            ax.set_ylabel('')
        despine(ax)

        ax.text(0.01, 0.95,
                f'Wilcoxon signed\nrank test\n-----------------------\nn={n}\nZ={z:.2f}\np={p:.3f}\nr={r:0.2f}\nMedian={diff.median().item():.2f}',
                ha='left', va='top', transform=ax.transAxes, fontsize=8)

        title = f'Evaluation set KGE difference: RS - {name}'
        
        ax.set_title(title, fontsize=8)

    fig.savefig('./figures/overfit.png', dpi=300, bbox_inches='tight')

overfit_test(ds_r, ds_h)